#  SILVER LAYER

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS ecommerce.silver_raw
""")

In [0]:
# ============================================================
# 06_SILVER
#
# Silver Layer Data Cleaning and Standardization
#
# Input:
#   ecommerce.silver
#
# Output:
#   ecommerce.silver
#
# Responsibilities:
#   - Remove unnecessary columns
#   - Fix data types
#   - Clean text fields
#   - Remove duplicates
#   - Handle invalid records
#
# CDC has already happened before this notebook.
# ============================================================


from pyspark.sql.functions import *



# ============================================================
# Configuration
# ============================================================

catalog = "ecommerce_prod"
catalog_v2 = "v2_prod"
catlogalog_v2 = "ecommerce"

schema = "silver"



# ============================================================
# 1. CUSTOMERS CLEANING
# ============================================================


customers = spark.table(
    f"{catalog}.{schema}.customers"
)



customers_clean = (

    customers


    # Remove CDC metadata if present

    .drop(
        "operation",
        "operation_timestamp",
        "_rescued_data"
    )


    # Remove records without primary key

    .filter(
        col("customer_id").isNotNull()
    )


    # Remove duplicates

    .dropDuplicates(
        ["customer_id"]
    )


    # Data type conversion

    .withColumn(
        "customer_id",
        col("customer_id").cast("int")
    )


    .withColumn(
        "loyalty_points",
        col("loyalty_points").cast("int")
    )


    .withColumn(
        "date_of_birth",
        to_date(col("date_of_birth"))
    )


    .withColumn(
        "registration_date",
        to_date(col("registration_date"))
    )


    # String cleaning

    .withColumn(
        "first_name",
        initcap(trim(col("first_name")))
    )


    .withColumn(
        "last_name",
        initcap(trim(col("last_name")))
    )


    .withColumn(
        "city",
        initcap(trim(col("city")))
    )


    .withColumn(
        "country",
        upper(trim(col("country")))
    )


    # Handle missing values

    .fillna(
        {
            "membership":"UNKNOWN",
            "status":"UNKNOWN"
        }
    )

)



customers_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "ecommerce.silver.customers"
    )


print("Customers Silver completed")





# ============================================================
# 2. PRODUCTS CLEANING
# ============================================================


products = spark.table(
    f"{catalog}.{schema}.products"
)



products_clean = (

    products


    .drop(
        "operation",
        "operation_timestamp",
        "_rescued_data"
    )


    .filter(
        col("product_id").isNotNull()
    )


    .dropDuplicates(
        ["product_id"]
    )


    .withColumn(
        "product_id",
        col("product_id").cast("int")
    )


    .withColumn(
        "price",
        col("price").cast("decimal(10,2)")
    )


    .withColumn(
        "cost_price",
        col("cost_price").cast("decimal(10,2)")
    )


    .withColumn(
        "stock_quantity",
        col("stock_quantity").cast("int")
    )


    .withColumn(
        "rating",
        col("rating").cast("double")
    )


    .withColumn(
        "product_name",
        trim(col("product_name"))
    )

)



products_clean.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable(
    "ecommerce.silver.products"
)



print("Products Silver completed")





# ============================================================
# 3. ORDERS CLEANING
# ============================================================


orders = spark.table(
    f"{catalog}.{schema}.orders"
)



# First, get valid customer and product IDs for referential integrity
valid_customer_ids = customers_clean.select("customer_id").distinct()
valid_product_ids = products_clean.select("product_id").distinct()


orders_clean = (

    orders


    .drop(
        "operation",
        "operation_timestamp",
        "_rescued_data"
    )


    .filter(
        col("order_id").isNotNull()
    )


    .dropDuplicates(
        ["order_id"]
    )


    # Filter out orders with invalid foreign keys
    .join(
        valid_customer_ids,
        "customer_id",
        "inner"
    )


    .join(
        valid_product_ids,
        "product_id",
        "inner"
    )


    .withColumn(
        "order_id",
        col("order_id").cast("int")
    )


    .withColumn(
        "customer_id",
        col("customer_id").cast("int")
    )


    .withColumn(
        "product_id",
        col("product_id").cast("int")
    )


    .withColumn(
        "quantity",
        col("quantity").cast("int")
    )


    .withColumn(
        "unit_price",
        col("unit_price").cast("decimal(10,2)")
    )


    .withColumn(
        "total_amount",
        col("total_amount").cast("decimal(10,2)")
    )


    .withColumn(
        "discount",
        col("discount").cast("decimal(10,2)")
    )


    .withColumn(
        "tax",
        col("tax").cast("decimal(10,2)")
    )


    .withColumn(
        "shipping_cost",
        col("shipping_cost").cast("decimal(10,2)")
    )


    .withColumn(
        "order_date",
        to_date(col("order_date"))
    )


    .withColumn(
        "city",
        initcap(trim(col("city")))
    )

)



orders_clean.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable(
    "ecommerce.silver.orders"
)



print("Orders Silver completed")





# ============================================================
# 4. PAYMENTS CLEANING
# ============================================================


payments = spark.table(
    f"{catalog}.{schema}.payments"
)



payments_clean = (

    payments


    .drop(
        "operation",
        "operation_timestamp",
        "_rescued_data"
    )


    .filter(
        col("payment_id").isNotNull()
    )


    .dropDuplicates(
        ["payment_id"]
    )


    .withColumn(
        "payment_id",
        col("payment_id").cast("int")
    )


    .withColumn(
        "order_id",
        col("order_id").cast("int")
    )


    .withColumn(
        "customer_id",
        col("customer_id").cast("int")
    )


    .withColumn(
        "amount",
        col("amount").cast("decimal(10,2)")
    )


    .withColumn(
        "payment_date",
        to_date(col("payment_date"))
    )


)



payments_clean.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable(
    "ecommerce.silver.payments"
)



print("Payments Silver completed")





# ============================================================
# 5. RETURNS CLEANING
# ============================================================


returns = spark.table(
    f"{catalog}.{schema}.returns"
)



returns_clean = (

    returns


    .drop(
        "operation",
        "operation_timestamp",
        "_rescued_data"
    )


    .filter(
        col("return_id").isNotNull()
    )


    .dropDuplicates(
        ["return_id"]
    )


    .withColumn(
        "return_id",
        col("return_id").cast("int")
    )


    .withColumn(
        "order_id",
        col("order_id").cast("int")
    )


    .withColumn(
        "customer_id",
        col("customer_id").cast("int")
    )


    .withColumn(
        "product_id",
        col("product_id").cast("int")
    )


    .withColumn(
        "refund_amount",
        col("refund_amount").cast("decimal(10,2)")
    )


    .withColumn(
        "return_date",
        to_date(col("return_date"))
    )


)



returns_clean.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable(
    "ecommerce.silver.returns"
)



print("Returns Silver completed")





# ============================================================
# Final Validation
# ============================================================


print("Silver Tables:")

spark.sql("""
SHOW TABLES IN ecommerce.silver
""").show()



for table in [
    "customers",
    "products",
    "orders",
    "payments",
    "returns"
]:

    print("\n")
    print("="*50)
    print(table.upper())
    print("="*50)

    display(
        spark.table(
            f"ecommerce.silver.{table}"
        ).limit(10)
    )

## Silver Schema evaluation

In [0]:
catalog = "ecommerce"
silver_schema = "silver"

datasets = [
    "customers",
    "products",
    "orders",
    "payments",
    "returns"
]
for dataset in datasets:

    print("\n")
    print("=" * 70)
    print(dataset.upper())
    print("=" * 70)

    spark.table(
        f"{catalog}.{silver_schema}.{dataset}"
    ).printSchema()